In [ ]:
import os

base_path = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition"

print(os.listdir(base_path))

In [ ]:
import os

base_path = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER"

print(os.listdir(base_path))

In [ ]:
import os

base_path = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER"

train_path = base_path + "/train/images"
test_path = base_path + "/test/images"

print("Train Images:", len(os.listdir(train_path)))
print("Test Images:", len(os.listdir(test_path)))

In [ ]:
import json
import os

ann_path = base_path + "/train/annotations"

json_files = os.listdir(ann_path)

print(json_files)

In [ ]:
import json
import os

json_file = "007.json"

with open(os.path.join(ann_path, json_file)) as f:
    data = json.load(f)

print(data)

In [ ]:
%pip install -q ultralytics

In [ ]:
import os

os.makedirs("/kaggle/working/datasets/images/train", exist_ok=True)
os.makedirs("/kaggle/working/datasets/labels/train", exist_ok=True)

print("Folders Created")

In [ ]:
import shutil
import os

base_path = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER"

train_img_path = base_path + "/train/images"

for img in os.listdir(train_img_path):

    shutil.copy(
        os.path.join(train_img_path, img),
        "/kaggle/working/datasets/images/train"
    )

print("Images Copied")

In [ ]:
import json

ann_path = base_path + "/train/annotations"

all_labels = set()

for jf in os.listdir(ann_path):

    with open(os.path.join(ann_path, jf)) as f:
        data = json.load(f)

    for shape in data["shapes"]:
        all_labels.add(shape["label"])

classes = sorted(list(all_labels))

class_to_id = {
    c:i for i,c in enumerate(classes)
}

print("Total Classes:", len(classes))

print(class_to_id)

In [ ]:
from PIL import Image
import json
import os
import glob

label_save_path = "/kaggle/working/datasets/labels/train"

for jf in os.listdir(ann_path):

    with open(os.path.join(ann_path, jf)) as f:
        data = json.load(f)

    image_id = jf.replace(".json", "")

    # Find image automatically
    img_files = glob.glob(
        os.path.join(train_img_path, image_id + ".*")
    )

    if len(img_files) == 0:
        print(f"Image not found for {image_id}")
        continue

    img_path = img_files[0]

    img = Image.open(img_path)

    img_w, img_h = img.size

    txt_name = jf.replace(".json", ".txt")

    lines = []

    for shape in data["shapes"]:

        label = shape["label"]

        class_id = class_to_id[label]

        (x1, y1), (x2, y2) = shape["points"]

        x_center = ((x1 + x2) / 2) / img_w
        y_center = ((y1 + y2) / 2) / img_h

        width = abs(x2 - x1) / img_w
        height = abs(y2 - y1) / img_h

        lines.append(
            f"{class_id} {x_center} {y_center} {width} {height}"
        )

    with open(os.path.join(label_save_path, txt_name), "w") as f:
        f.write("\n".join(lines))

print("YOLO Labels Created")

In [ ]:
data_yaml = f"""
train: /kaggle/working/datasets/images/train

val: /kaggle/working/datasets/images/train

nc: {len(classes)}

names: {classes}
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    augment=True
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")

results = model.predict(
    source=base_path + "/test/images",
    conf=0.00001,
    iou=0.1,
    save=True,
    max_det=3000
)

print("Prediction Complete")

In [ ]:
import pandas as pd

submission = []

for r in results:

    image_name = r.path.split("/")[-1]

    preds = []

    boxes = r.boxes.xyxy.cpu().numpy()
    classes_pred = r.boxes.cls.cpu().numpy()

    for box, cls_id in zip(boxes, classes_pred):

        x1, y1, x2, y2 = box

        label = classes[int(cls_id)]

        pred = f"{label} {x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f}"

        preds.append(pred)

    prediction_string = " ".join(preds)

    submission.append(
        [image_name, prediction_string]
    )

df = pd.DataFrame(
    submission,
    columns=["image_id", "prediction"]
)

df.to_csv("submission.csv", index=False)

print(df.head())

In [ ]:
import os

print(os.listdir("/kaggle/working/runs/detect/predict-3"))

In [ ]:
from IPython.display import Image, display
import os

pred_path = "/kaggle/working/runs/detect/predict-3"

for img in os.listdir(pred_path):

    print(img)

    display(
        Image(filename=os.path.join(pred_path, img))
    )

In [ ]:
results = model.predict(
    source=base_path + "/test/images",
    conf=0.02,
    iou=0.3,
    save=True,
    max_det=300
)

print("Prediction Complete")

In [ ]:
import shutil
import os

label_dir = "/kaggle/working/datasets/labels/train"

shutil.rmtree(label_dir)

os.makedirs(label_dir, exist_ok=True)

print("Old labels removed")

In [ ]:
from PIL import Image
import json
import os
import glob

label_save_path = "/kaggle/working/datasets/labels/train"

for jf in os.listdir(ann_path):

    with open(os.path.join(ann_path, jf)) as f:
        data = json.load(f)

    image_id = jf.replace(".json", "")

    img_files = glob.glob(
        os.path.join(train_img_path, image_id + ".*")
    )

    if len(img_files) == 0:
        continue

    img_path = img_files[0]

    img = Image.open(img_path)

    img_w, img_h = img.size

    txt_name = jf.replace(".json", ".txt")

    lines = []

    for shape in data["shapes"]:

        (x1, y1), (x2, y2) = shape["points"]

        x_center = ((x1 + x2) / 2) / img_w
        y_center = ((y1 + y2) / 2) / img_h

        width = abs(x2 - x1) / img_w
        height = abs(y2 - y1) / img_h

        # SINGLE CLASS
        lines.append(
            f"0 {x_center} {y_center} {width} {height}"
        )

    with open(os.path.join(label_save_path, txt_name), "w") as f:
        f.write("\n".join(lines))

print("Single-class YOLO labels created")

In [ ]:
data_yaml = """
train: /kaggle/working/datasets/images/train

val: /kaggle/working/datasets/images/train

nc: 1

names: ['char']
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    augment=True
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/train-2/weights/best.pt")
results = model.predict(
    source=base_path + "/test/images",
    conf=0.20,
    iou=0.45,
    save=True,
    max_det=80
)

print("Prediction Complete")

In [ ]:
from IPython.display import Image, display
import os

pred_path = "/kaggle/working/runs/detect/predict-6"

for img in os.listdir(pred_path):

    print(img)

    display(
        Image(filename=os.path.join(pred_path, img))
    )

## Two-stage character segmentation + recognition

The goal of this notebook is to predict every handwritten character in each test image. For each character, the final prediction should contain two things:

1. Where the character is located in the image.
2. What character it is.

This is why the solution is split into two smaller problems instead of trying to solve everything with one model.

### Problem 1: character segmentation

Segmentation/localization means finding the position of each character in the image. In this notebook, YOLO is used for this part.

YOLO looks at the full image and predicts bounding boxes around characters. Each box has four coordinates:

```text
x1 y1 x2 y2
```

These coordinates mean:

- `x1`: left side of the box
- `y1`: top side of the box
- `x2`: right side of the box
- `y2`: bottom side of the box

For segmentation, the model does not need to know the exact character label. It only needs to learn this question:

```text
Is there a character here?
```

That is why the YOLO detector is trained as a single-class model with the class name `char`. This makes YOLO focus only on finding all character regions.

### Problem 2: character recognition

Recognition means deciding what the detected character actually is. For example, after YOLO finds a box, the classifier should decide whether the crop is `ka`, `kha`, `0`, `1`, `A`, or any other label present in the training annotations.

The classifier does not look at the full image. It only looks at small cropped images of individual characters. These crops are created from the training annotation boxes.

So the recognition model learns this question:

```text
This cropped character image belongs to which class?
```

### Why use two models?

Using two stages makes the task easier:

- YOLO becomes responsible for finding every character location.
- The classifier becomes responsible for identifying the character inside each box.
- The detector does not need to learn many character classes at the same time.
- The classifier receives cleaner inputs because each image crop contains mostly one character.

This is especially useful for handwritten multiscript data, where many characters may look similar and images may contain multiple characters.

### How the new cells work

#### Stage 2A: build character crops

This cell reads the training annotation JSON files. Each annotation contains character labels and box points.

For every annotated character, the cell:

1. Opens the original training image.
2. Reads the character box from the JSON annotation.
3. Crops that character region from the image.
4. Saves the crop into a folder for its class.
5. Creates a `class_mapping.json` file so numeric class IDs can be converted back to real labels.

The crop dataset is saved here:

```text
/kaggle/working/char_crops
```

It is split into:

```text
/kaggle/working/char_crops/train
/kaggle/working/char_crops/val
```

#### Stage 2B: train the character classifier

This cell trains a small CNN classifier using the cropped character images.

The classifier receives one crop at a time and predicts the character class. The image is converted to grayscale, resized to `64 x 64`, normalized, and then passed through the CNN.

The cell also uses a weighted sampler. This helps when some characters appear more often than others. Without this, the model may become biased toward common characters.

The best classifier checkpoint is saved here:

```text
/kaggle/working/char_classifier.pt
```

#### Stage 2C: combine YOLO and the classifier

This is the final prediction step.

For each test image:

1. YOLO predicts character boxes.
2. Each detected box is cropped from the image.
3. The classifier predicts the label of each crop.
4. The label and box coordinates are joined together.
5. The final prediction string is written to a CSV file.

The final output format is:

```text
label x1 y1 x2 y2 label x1 y1 x2 y2 ...
```

For example:

```text
ka 12.40 18.20 35.90 49.10 kha 42.00 17.50 63.20 48.80
```

The final CSV is saved here:

```text
/kaggle/working/submission_two_stage.csv
```

### Order to run the notebook

Run the notebook in this order:

1. Install and import dependencies.
2. Prepare the dataset paths.
3. Train the single-class YOLO detector using `names: ['char']`.
4. Run Stage 2A to create cropped character images.
5. Run Stage 2B to train the character classifier.
6. Run Stage 2C to generate the final submission CSV.

### Important idea

YOLO answers:

```text
Where is each character?
```

The classifier answers:

```text
What is this character?
```

Together, they solve the complete task:

```text
Find every character and identify it.
```

In [ ]:
# Stage 2A: build a labeled crop dataset for character recognition
from pathlib import Path
from PIL import Image
import glob
import json
import os
import random
import shutil

base_path = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER"
train_img_path = os.path.join(base_path, "train/images")
ann_path = os.path.join(base_path, "train/annotations")

crop_root = Path("/kaggle/working/char_crops")
for split in ["train", "val"]:
    split_dir = crop_root / split
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)

json_files = sorted([f for f in os.listdir(ann_path) if f.endswith(".json")])
all_labels = sorted({
    shape["label"]
    for jf in json_files
    for shape in json.load(open(os.path.join(ann_path, jf), encoding="utf-8"))["shapes"]
})
class_to_id = {label: idx for idx, label in enumerate(all_labels)}
id_to_class = {idx: label for label, idx in class_to_id.items()}

for split in ["train", "val"]:
    for class_id in range(len(all_labels)):
        (crop_root / split / f"{class_id:04d}").mkdir(parents=True, exist_ok=True)

random.seed(42)
random.shuffle(json_files)
val_count = max(1, int(0.15 * len(json_files))) if len(json_files) > 1 else 0
val_files = set(json_files[:val_count])

crop_counts = {"train": 0, "val": 0}

for jf in json_files:
    with open(os.path.join(ann_path, jf), encoding="utf-8") as f:
        data = json.load(f)

    image_id = jf.replace(".json", "")
    img_files = glob.glob(os.path.join(train_img_path, image_id + ".*"))
    if not img_files:
        print(f"Image not found for {image_id}")
        continue

    image = Image.open(img_files[0]).convert("RGB")
    img_w, img_h = image.size
    split = "val" if jf in val_files else "train"

    for box_idx, shape in enumerate(data["shapes"]):
        label = shape["label"]
        class_id = class_to_id[label]
        points = shape["points"]
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]

        x1, x2 = min(xs), max(xs)
        y1, y2 = min(ys), max(ys)
        pad = max(2, int(0.05 * max(x2 - x1, y2 - y1)))
        x1 = max(0, int(round(x1)) - pad)
        y1 = max(0, int(round(y1)) - pad)
        x2 = min(img_w, int(round(x2)) + pad)
        y2 = min(img_h, int(round(y2)) + pad)

        if x2 <= x1 or y2 <= y1:
            continue

        class_dir = crop_root / split / f"{class_id:04d}"
        crop = image.crop((x1, y1, x2, y2))
        crop.save(class_dir / f"{image_id}_{box_idx:03d}.png")
        crop_counts[split] += 1

mapping = {
    "id_to_class": {str(idx): label for idx, label in id_to_class.items()},
    "class_to_id": class_to_id,
}
with open(crop_root / "class_mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print("Classes:", len(all_labels))
print("Crop counts:", crop_counts)
print("Crop dataset:", crop_root)

In [ ]:
# Stage 2B: train a character classifier on the crops
from pathlib import Path
from PIL import Image
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

crop_root = Path("/kaggle/working/char_crops")
with open(crop_root / "class_mapping.json", encoding="utf-8") as f:
    mapping = json.load(f)

num_classes = len(mapping["id_to_class"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.RandomAffine(degrees=8, translate=(0.08, 0.08), scale=(0.9, 1.1), shear=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
val_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

class CropDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        for class_dir in sorted(self.root.iterdir()):
            if not class_dir.is_dir():
                continue
            class_id = int(class_dir.name)
            for image_path in sorted(class_dir.glob("*.png")):
                self.samples.append((image_path, class_id))
        if not self.samples:
            raise ValueError(f"No crop images found in {self.root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, class_id = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, class_id

train_ds = CropDataset(crop_root / "train", transform=train_tfms)
val_ds = CropDataset(crop_root / "val", transform=val_tfms)

train_targets = torch.tensor([class_id for _, class_id in train_ds.samples], dtype=torch.long)
class_counts = torch.bincount(train_targets, minlength=num_classes).float().clamp_min(1)
sample_weights = 1.0 / class_counts[train_targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

class CharCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

classifier = CharCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_val_acc = 0.0
epochs = 15
checkpoint_path = "/kaggle/working/char_classifier.pt"

for epoch in range(epochs):
    classifier.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = classifier(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    classifier.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = classifier(images)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()
    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)
    avg_loss = train_loss / max(train_total, 1)
    print(f"Epoch {epoch + 1:02d}/{epochs} | loss {avg_loss:.4f} | train_acc {train_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": classifier.state_dict(),
            "id_to_class": mapping["id_to_class"],
            "num_classes": num_classes,
        }, checkpoint_path)

print("Best validation accuracy:", best_val_acc)
print("Saved classifier:", checkpoint_path)

In [ ]:
# Stage 2C: combine YOLO segmentation with the crop classifier
from pathlib import Path
from PIL import Image
import glob
import os
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from ultralytics import YOLO

base_path = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER"
test_img_path = os.path.join(base_path, "test/images")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class CharCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

classifier_checkpoint = torch.load("/kaggle/working/char_classifier.pt", map_location=device)
id_to_class = {int(k): v for k, v in classifier_checkpoint["id_to_class"].items()}
classifier = CharCNN(classifier_checkpoint["num_classes"]).to(device)
classifier.load_state_dict(classifier_checkpoint["model_state"])
classifier.eval()

classifier_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

detector_weights = sorted(
    glob.glob("/kaggle/working/runs/detect/train*/weights/best.pt"),
    key=os.path.getmtime,
)
if not detector_weights:
    raise FileNotFoundError("Train the single-class YOLO character detector before running this cell.")

detector = YOLO(detector_weights[-1])
print("Detector weights:", detector_weights[-1])

results = detector.predict(
    source=test_img_path,
    conf=0.20,
    iou=0.45,
    save=True,
    max_det=120,
)

def recognize_crop(crop):
    tensor = classifier_tfms(crop).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = classifier(tensor)
        pred_id = int(logits.argmax(1).item())
    return id_to_class[pred_id]

submission = []

for result in results:
    image_name = os.path.basename(result.path)
    image = Image.open(result.path).convert("RGB")
    img_w, img_h = image.size
    predictions = []

    boxes = result.boxes.xyxy.cpu().numpy()
    for box in boxes:
        x1, y1, x2, y2 = [float(v) for v in box]
        x1_i = max(0, min(img_w - 1, int(round(x1))))
        y1_i = max(0, min(img_h - 1, int(round(y1))))
        x2_i = max(1, min(img_w, int(round(x2))))
        y2_i = max(1, min(img_h, int(round(y2))))

        if x2_i <= x1_i or y2_i <= y1_i:
            continue

        crop = image.crop((x1_i, y1_i, x2_i, y2_i))
        label = recognize_crop(crop)
        predictions.append({
            "label": label,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

    predictions.sort(key=lambda pred: ((pred["y1"] + pred["y2"]) / 2, pred["x1"]))
    prediction_string = " ".join(
        f"{pred['label']} {pred['x1']:.2f} {pred['y1']:.2f} {pred['x2']:.2f} {pred['y2']:.2f}"
        for pred in predictions
    )
    submission.append([image_name, prediction_string])

df = pd.DataFrame(submission, columns=["image_id", "prediction"])
df.to_csv("/kaggle/working/submission_two_stage.csv", index=False)
print(df.head())
print("Saved:", "/kaggle/working/submission_two_stage.csv")